In [ ]:
!pip install openai pandas tqdm -q

In [ ]:
import openai
import pandas as pd
from tqdm import tqdm
import time
import os

In [ ]:
YANDEX_CLOUD_FOLDER = "b1guceu80lgn199hh63p"
YANDEX_CLOUD_API_KEY = "" # api-ключ
YANDEX_CLOUD_MODEL = "yandexgpt-5.1/latest"

In [ ]:
TEMPERATURE = 1.0 # нужное значение температуры
ATTEMPT = 3 # номер попытки

In [ ]:
OUT_FILE = "yandexgpt_all_results.csv"

In [ ]:
file_name = "for_model.csv"
df = pd.read_csv(file_name)

sentences = df['Sentences'].dropna().tolist()

In [ ]:
system_prompt = """Исправьте ошибки в предложении. Не меняйте смысл.
Вносите минимальные необходимые правки.
Если ошибок нет — верните предложение без изменений.
Отвечайте только исправленным текстом, без пояснений."""

def user_prompt(sentence):
    return f"Предложение: {sentence}"

In [ ]:
def correct_sentence(sentence, temperature):
    client = openai.OpenAI(
        api_key=YANDEX_CLOUD_API_KEY,
        base_url="https://ai.api.cloud.yandex.net/v1",
        project=YANDEX_CLOUD_FOLDER
    )
    try:
        response = client.responses.create(
            model=f"gpt://{YANDEX_CLOUD_FOLDER}/{YANDEX_CLOUD_MODEL}",
            temperature=temperature,
            instructions=system_prompt,
            input=user_prompt(sentence),
            max_output_tokens=500
        )
        return response.output_text.strip()
    except Exception as e:
        return f"ОШИБКА: {str(e)}"


In [ ]:
results = []
for sent in tqdm(sentences, desc="Исправление"):
    corrected = correct_sentence(sent, TEMPERATURE)
    results.append({
        "original": sent,
        "corrected": corrected,
        "temperature": TEMPERATURE,
        "attempt": ATTEMPT
    })
    time.sleep(0.5)

Исправление: 100%|██████████| 32/32 [00:51<00:00,  1.61s/it]


In [ ]:
new_rows = pd.DataFrame(results)

if os.path.exists(OUT_FILE):
    existing_df = pd.read_csv(OUT_FILE)
    combined_df = pd.concat([existing_df, new_rows], ignore_index=True)
    combined_df.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
else:
    new_rows.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")